
# `fuente_unica.ipynb` — Deduplicación por razón social

**Objetivo.** Deduplicar `Base_exportadores_frutas_a_Col.xlsx` usando **solo** la variable `DESC RAZON EXPORTADOR` como señal de identidad. La columna `PAIS` queda preparada como extensión opcional, pero por defecto **no participa** en el scoring.

**Decisión técnica.** `rues-linker` exige una columna de NIT para su contrato interno. En este notebook se crea una columna técnica `NIT` vacía; no aporta señal. El perfil usado es `deduplication_sin_nit_conservador`, que fue diseñado para fuentes sin NIT y usa razón social limpia, blocking LSH, similitud textual, fonética derivada del nombre e IDF de tokens.

**Ruta esperada en Colab.** El Excel debe estar en:

`/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29/Base_exportadores_frutas_a_Col.xlsx`

**Salida principal.** El notebook genera una carpeta de corrida con: correlativa, golden de exportadores deduplicados, conexiones no triviales, QA de clusters y un Excel consolidado con varias hojas.



## § A — Setup base

Ejecute estas celdas una vez por sesión. La configuración editable está concentrada en la celda B.


In [1]:

# ════════════════════════════════════════════════════════════════════════
# A.1 — Imports + Drive mount + logging (idempotente)
# ════════════════════════════════════════════════════════════════════════

from __future__ import annotations

import contextlib
import gc
import importlib
import io
import json
import logging
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import psutil
except Exception:  # pragma: no cover
    psutil = None

try:
    from google.colab import drive  # type: ignore
    EN_COLAB = True
except Exception:
    drive = None
    EN_COLAB = False


def montar_drive_con_retry(intentos: int = 3, espera_s: int = 5) -> bool:
    """Monta Google Drive en Colab con reintentos y sin remonte innecesario."""
    if not EN_COLAB:
        print("ℹ️  Entorno local detectado; no se monta Google Drive.")
        return False
    if os.path.ismount("/content/drive"):
        print("ℹ️  Drive ya estaba montado.")
        return True
    for intento in range(1, intentos + 1):
        try:
            drive.mount("/content/drive")
            print(f"✅ Drive montado (intento {intento}/{intentos}).")
            return True
        except Exception as exc:
            print(f"⚠️  Intento {intento}/{intentos} falló: {type(exc).__name__}: {exc}")
            if intento < intentos:
                time.sleep(espera_s)
            else:
                raise
    return False


def setup_logger(log_file: Path | None = None, level: int = logging.INFO) -> logging.Logger:
    """Logger dual: stdout + archivo opcional. Se reinicia de forma idempotente."""
    logger = logging.getLogger("fuente_unica")
    logger.setLevel(level)
    logger.handlers.clear()

    fmt = logging.Formatter(
        "%(asctime)s [%(levelname)-7s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    if log_file is not None:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)

    logger.propagate = False
    return logger


montar_drive_con_retry()
log = setup_logger()
print(f"🐍 Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"📦 pandas {pd.__version__} · numpy {np.__version__}")
print(f"🧭 EN_COLAB={EN_COLAB}")


ℹ️  Drive ya estaba montado.
🐍 Python 3.12.13
📦 pandas 2.2.2 · numpy 2.0.2
🧭 EN_COLAB=True


In [2]:

# ════════════════════════════════════════════════════════════════════════
# A.2 — Dataclasses y helpers de rutas
# ════════════════════════════════════════════════════════════════════════


def _default_ruta_paquete() -> Path:
    if EN_COLAB:
        return Path("/content/drive/MyDrive/ProColombia/record_linkage")
    return Path("/mnt/data/rues_unpacked/rues-linker-main")


def _default_dir_base() -> Path:
    if EN_COLAB:
        return Path("/content/drive/MyDrive/ProColombia/Deduplicación")
    return Path("/mnt/data")


def _default_dir_datos() -> Path:
    if EN_COLAB:
        return Path("/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29")
    return Path("/mnt/data")


@dataclass(frozen=True)
class ConfigFuenteUnica:
    """Configuración de la corrida. Edite esto en la celda B, no en helpers."""

    # Identidad de corrida
    fecha_procesamiento: str = "2026-06-29"
    etiqueta_corrida: str = "FUENTE_UNICA"
    seed: int = 42

    # Rutas
    ruta_paquete: Path = field(default_factory=_default_ruta_paquete)
    dir_base: Path = field(default_factory=_default_dir_base)
    dir_datos: Path = field(default_factory=_default_dir_datos)
    archivo_excel: str = "Base_exportadores_frutas_a_Col.xlsx"
    hoja_excel: int | str = 0

    # Fuente / columnas
    nombre_fuente: str = "EXPORTADORES_FRUTAS_COL"
    col_nombre_original: str = "DESC RAZON EXPORTADOR"
    col_nombre_estandar: str = "RAZON_SOCIAL"
    col_nit_tecnico: str = "NIT"
    col_pais: str = "PAIS"

    # Contrato del pipeline
    cleaning_mode: str = "BALANCEADO"
    perfil_base: str = "deduplication_sin_nit_conservador"
    perfil_runtime: str = "fuente_unica_nombre_conservador"
    perfil_overrides: dict[str, Any] = field(
        default_factory=lambda: {
            # Valores explícitos del perfil sin-NIT conservador.
            # Están aquí para que sean auditables y editables sin tocar la librería.
            "lsh_threshold": 0.50,
            "score_threshold": 0.80,
            "min_name_similarity": 0.75,
            "remove_top_words": 20,
            "idf_weight_blend": 0.50,
            "use_strict_clusters": True,
        }
    )

    # Extensión futura: por defecto PAIS NO participa.
    usar_pais_en_scoring: bool = False
    extra_features_pais: tuple[dict[str, Any], ...] = (
        {"column": "PAIS", "weight": 0.20, "type": "categorical_signed"},
    )

    # Ejecución / QA
    eliminar_nombre_vacio: bool = True
    smoke_first: bool = True
    smoke_n: int = 300
    silenciar_stdout_pipeline: bool = False
    max_variantes_por_celda: int = 12
    max_qa_clusters: int = 200

    # Exportación
    exportar_csv: bool = True
    exportar_parquet: bool = True
    exportar_excel: bool = True

    def __post_init__(self) -> None:
        object.__setattr__(self, "ruta_paquete", Path(self.ruta_paquete))
        object.__setattr__(self, "dir_base", Path(self.dir_base))
        object.__setattr__(self, "dir_datos", Path(self.dir_datos))
        if not self.fecha_procesamiento:
            raise ValueError("fecha_procesamiento no puede estar vacía.")
        if not self.col_nombre_original:
            raise ValueError("Debe declarar col_nombre_original.")
        if self.usar_pais_en_scoring and not self.col_pais:
            raise ValueError("Si usar_pais_en_scoring=True, declare col_pais.")


@dataclass(frozen=True)
class RutasFuenteUnica:
    """Single source of truth para los artefactos de la corrida."""

    config: ConfigFuenteUnica
    timestamp: str = field(default_factory=lambda: datetime.now().strftime("%Y%m%d_%H%M%S"))

    @property
    def nombre_ejecucion(self) -> str:
        return f"{self.config.etiqueta_corrida}_{self.config.fecha_procesamiento}_{self.timestamp}"

    @property
    def archivo_excel(self) -> Path:
        return self.config.dir_datos / self.config.archivo_excel

    @property
    def run_dir(self) -> Path:
        return self.config.dir_base / "salidas" / self.config.fecha_procesamiento / self.nombre_ejecucion

    @property
    def logs(self) -> Path:
        return self.run_dir / "logs"

    @property
    def intermedios(self) -> Path:
        return self.run_dir / "intermedios_rues_linker"

    @property
    def qa(self) -> Path:
        return self.run_dir / "qa"

    @property
    def exports(self) -> Path:
        return self.run_dir / "exports"

    def crear(self) -> None:
        for p in (self.run_dir, self.logs, self.intermedios, self.qa, self.exports):
            p.mkdir(parents=True, exist_ok=True)


TIEMPOS: dict[str, float] = {}


def fin_paso(nombre: str, t0: float) -> None:
    TIEMPOS[nombre] = round(time.time() - t0, 2)
    print(f"\n⏱️  {nombre}: {TIEMPOS[nombre]:,.2f} s")


def memoria_actual() -> str:
    if psutil is None:
        return "psutil no disponible"
    proc = psutil.Process(os.getpid())
    rss = proc.memory_info().rss / 1024**3
    vm = psutil.virtual_memory()
    return f"proceso={rss:.2f} GB | sistema={vm.percent:.1f}% usado | libre={vm.available/1024**3:.2f} GB"


In [3]:

# ════════════════════════════════════════════════════════════════════════
# A.3 — Utilidades: instalación, lectura, ejecución, QA y exportación
# ════════════════════════════════════════════════════════════════════════


def resolver_repo_rues_linker(ruta_paquete: Path) -> dict[str, Any]:
    """Resuelve cómo activar rues-linker desde una ruta de Drive.

    Acepta:
      1) Repo instalable con pyproject.toml.
      2) Carpeta padre que contiene record_linkage/ directamente.
      3) Carpeta record_linkage/ directa.
      4) Carpeta que contiene rues-linker-main/.
      5) Carpeta o archivo zip rues-linker*.zip.
    """
    ruta_paquete = Path(ruta_paquete)
    candidatos = [
        ruta_paquete,
        ruta_paquete / "rues-linker-main",
        ruta_paquete / "rues-linker",
    ]

    # Caso moderno: repo con pyproject.toml.
    for cand in candidatos:
        if (cand / "pyproject.toml").exists():
            return {"kind": "repo", "path": cand}

    # Caso legacy: carpeta padre que contiene record_linkage/.
    for cand in candidatos:
        if (cand / "record_linkage" / "__init__.py").exists():
            return {"kind": "src_parent", "path": cand}

    # Caso legacy: la ruta ES la carpeta record_linkage/.
    if (ruta_paquete / "__init__.py").exists() and ruta_paquete.name == "record_linkage":
        return {"kind": "package_dir", "path": ruta_paquete.parent}

    # Caso zip.
    zips: list[Path] = []
    if ruta_paquete.is_file() and ruta_paquete.suffix.lower() == ".zip":
        zips.append(ruta_paquete)
    elif ruta_paquete.exists():
        zips.extend(sorted(ruta_paquete.glob("rues-linker*.zip")))
        zips.extend(sorted(ruta_paquete.glob("*rues*linker*.zip")))

    if zips:
        zip_path = zips[0]
        extract_dir = Path("/content/rues_linker_src") if EN_COLAB else Path("/tmp/rues_linker_src")
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        for cand in [extract_dir, *extract_dir.iterdir()]:
            if (cand / "pyproject.toml").exists():
                return {"kind": "repo", "path": cand}
            if (cand / "record_linkage" / "__init__.py").exists():
                return {"kind": "src_parent", "path": cand}

    raise FileNotFoundError(
        "No pude resolver rues-linker. Ruta recibida: "
        f"{ruta_paquete}. Debe contener pyproject.toml, record_linkage/ o un zip rues-linker*.zip."
    )


def instalar_rues_linker(ruta_paquete: Path) -> Path:
    """Instala o activa rues-linker; prioriza repo editable si existe."""
    info = resolver_repo_rues_linker(ruta_paquete)
    kind = info["kind"]
    root = Path(info["path"])

    if kind == "repo":
        cmd = [sys.executable, "-m", "pip", "install", "-q", "-e", f"{root}[excel]"]
        print("▶ Instalando repo editable:", " ".join(cmd))
        subprocess.run(cmd, check=True)
        for cand in (root / "src", root):
            if (cand / "record_linkage" / "__init__.py").exists() and str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
    else:
        # Fallback para layouts antiguos sin pyproject.toml.
        print(f"▶ Activando paquete local legacy: {root}")
        deps = [
            "pyarrow>=14.0", "scikit-learn>=1.3", "scipy>=1.10", "datasketch>=1.6",
            "rapidfuzz>=3.0", "unidecode>=1.3", "tqdm>=4.66", "pydantic>=2.0",
            "pyyaml>=6.0", "openpyxl>=3.1", "xlsxwriter>=3.1",
        ]
        if EN_COLAB:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))

    importlib.invalidate_caches()
    import record_linkage
    print("✅ record_linkage importado desde:", Path(record_linkage.__file__).resolve())
    return root


def configurar_perfil_runtime(config: ConfigFuenteUnica) -> dict[str, Any]:
    """Crea un perfil runtime sin modificar archivos fuente del paquete."""
    from record_linkage.pipeline._internal import DEDUPLICATION_PROFILES

    if config.perfil_base not in DEDUPLICATION_PROFILES:
        raise KeyError(
            f"Perfil base no existe: {config.perfil_base}. "
            f"Disponibles: {sorted(DEDUPLICATION_PROFILES)}"
        )
    perfil = DEDUPLICATION_PROFILES[config.perfil_base].copy()
    perfil.update(config.perfil_overrides)
    if config.usar_pais_en_scoring:
        perfil["extra_features"] = [dict(x) for x in config.extra_features_pais]
    else:
        perfil.pop("extra_features", None)
    DEDUPLICATION_PROFILES[config.perfil_runtime] = perfil
    return perfil


def resolver_archivo_excel(rutas: RutasFuenteUnica) -> Path:
    """Ubica el Excel en la ruta configurada o en fallbacks útiles para pruebas."""
    candidatos = [
        rutas.archivo_excel,
        Path("/content") / rutas.config.archivo_excel,
        Path.cwd() / rutas.config.archivo_excel,
        Path("/mnt/data") / rutas.config.archivo_excel,
    ]
    for cand in candidatos:
        if cand.exists():
            return cand
    raise FileNotFoundError(
        "No encontré el Excel. Rutas probadas:\n" + "\n".join(f"  • {c}" for c in candidatos)
    )


def cargar_y_preparar_base(config: ConfigFuenteUnica, rutas: RutasFuenteUnica) -> pd.DataFrame:
    """Lee Excel, valida columna de nombre y prepara contrato mínimo para rues-linker."""
    excel_path = resolver_archivo_excel(rutas)
    print(f"📥 Leyendo Excel: {excel_path}")
    df = pd.read_excel(excel_path, sheet_name=config.hoja_excel, dtype=str)
    df.columns = [str(c).strip() for c in df.columns]

    if config.col_nombre_original not in df.columns:
        raise KeyError(
            f"No existe la columna {config.col_nombre_original!r}. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    df = df.copy()
    df["_ROW_ID_ORIGINAL"] = np.arange(len(df), dtype="int64")
    df["SRC"] = config.nombre_fuente
    df[config.col_nombre_estandar] = df[config.col_nombre_original].astype("string")
    df[config.col_nit_tecnico] = ""  # Columna técnica: NO aporta señal de identidad.

    if config.eliminar_nombre_vacio:
        mask_nombre = df[config.col_nombre_estandar].notna() & df[config.col_nombre_estandar].str.strip().ne("")
        eliminados = int((~mask_nombre).sum())
        df = df.loc[mask_nombre].copy()
        if eliminados:
            print(f"⚠️  Se excluyeron {eliminados:,} filas sin {config.col_nombre_original}.")

    print(f"✅ Base preparada: {len(df):,} registros · {df.shape[1]:,} columnas")
    print(f"🔎 Nombres exactos únicos: {df[config.col_nombre_original].nunique(dropna=True):,}")
    if config.col_pais in df.columns:
        print(f"🌎 Países únicos informativos: {df[config.col_pais].nunique(dropna=True):,} (no usados en scoring por defecto)")
    return df


def ejecutar_deduplicacion(
    df: pd.DataFrame,
    config: ConfigFuenteUnica,
    output_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Ejecuta deduplicate_unified con opción de capturar stdout verboso."""
    from record_linkage.deduplication.unified import deduplicate_unified

    kwargs = dict(
        df_input=df,
        col_nit=config.col_nit_tecnico,
        col_name=config.col_nombre_estandar,
        mode=config.cleaning_mode,
        profile=config.perfil_runtime,
        output_dir=str(output_dir),
        validate_against_legacy=False,
        extra_features=[dict(x) for x in config.extra_features_pais] if config.usar_pais_en_scoring else None,
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    if config.silenciar_stdout_pipeline:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            correlativa, conexiones = deduplicate_unified(**kwargs)
        (output_dir / "stdout_pipeline.txt").write_text(buf.getvalue(), encoding="utf-8")
    else:
        correlativa, conexiones = deduplicate_unified(**kwargs)
    return correlativa, conexiones


def _colapsar_unicos(s: pd.Series, max_items: int = 12) -> str:
    vals = (
        s.dropna()
        .astype(str)
        .map(str.strip)
        .replace("", np.nan)
        .dropna()
        .drop_duplicates()
        .tolist()
    )
    vals = sorted(vals)
    if len(vals) > max_items:
        return " | ".join(vals[:max_items]) + f" | ... (+{len(vals) - max_items})"
    return " | ".join(vals)


def construir_golden_exportadores(correlativa: pd.DataFrame, config: ConfigFuenteUnica) -> pd.DataFrame:
    """Construye una tabla golden: una fila por exportador deduplicado."""
    df = correlativa.copy()
    df["CLUSTER_SIZE"] = df.groupby("ID_GRUPO")["ID_GRUPO"].transform("size")

    kg_cols = [c for c in df.columns if re.fullmatch(r"KG NETO \d{4}", str(c))]
    for col in kg_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    agg_dict: dict[str, tuple[str, Any]] = {
        "DESC_RAZON_EXPORTADOR_CANONICA": ("RAZON_SOCIAL_FINAL", lambda x: _colapsar_unicos(x, 1)),
        "REGISTROS_ORIGINALES": ("ID_GRUPO", "size"),
        "VARIANTES_RAZON_SOCIAL": (config.col_nombre_original, lambda x: _colapsar_unicos(x, config.max_variantes_por_celda)),
        "N_VARIANTES_RAZON_SOCIAL": (config.col_nombre_original, lambda x: x.dropna().astype(str).str.strip().nunique()),
        "FILAS_ORIGINALES": ("_ROW_ID_ORIGINAL", lambda x: _colapsar_unicos(x.astype(str), config.max_variantes_por_celda)),
    }
    if config.col_pais in df.columns:
        agg_dict["PAISES_INFORMATIVOS"] = (config.col_pais, lambda x: _colapsar_unicos(x, config.max_variantes_por_celda))
        agg_dict["N_PAISES_INFORMATIVOS"] = (config.col_pais, lambda x: x.dropna().astype(str).str.strip().nunique())
    if "DESCRIPCION ARANCELARIA" in df.columns:
        agg_dict["DESCRIPCIONES_ARANCELARIAS"] = (
            "DESCRIPCION ARANCELARIA",
            lambda x: _colapsar_unicos(x, config.max_variantes_por_celda),
        )
        agg_dict["N_DESCRIPCIONES_ARANCELARIAS"] = (
            "DESCRIPCION ARANCELARIA",
            lambda x: x.dropna().astype(str).str.strip().nunique(),
        )

    golden = df.groupby("ID_GRUPO", as_index=False).agg(**agg_dict)
    if kg_cols:
        suma_kg = df.groupby("ID_GRUPO", as_index=False)[kg_cols].sum(min_count=1)
        golden = golden.merge(suma_kg, on="ID_GRUPO", how="left")

    golden = golden.sort_values(
        ["REGISTROS_ORIGINALES", "DESC_RAZON_EXPORTADOR_CANONICA"],
        ascending=[False, True],
    ).reset_index(drop=True)
    return golden


def construir_qa_clusters(correlativa: pd.DataFrame, config: ConfigFuenteUnica) -> pd.DataFrame:
    """Genera QA de clusters no triviales para revisión humana."""
    try:
        from rapidfuzz import fuzz
    except Exception:
        fuzz = None

    df = correlativa.copy()
    df["CLUSTER_SIZE"] = df.groupby("ID_GRUPO")["ID_GRUPO"].transform("size")
    rows: list[dict[str, Any]] = []

    for gid, g in df.loc[df["CLUSTER_SIZE"] > 1].groupby("ID_GRUPO", sort=False):
        nombres = (
            g[config.col_nombre_original]
            .dropna()
            .astype(str)
            .map(str.strip)
            .drop_duplicates()
            .tolist()
        )
        min_sim = None
        avg_sim = None
        if fuzz is not None and len(nombres) >= 2:
            sims = []
            for i in range(len(nombres)):
                for j in range(i + 1, len(nombres)):
                    sims.append(fuzz.token_set_ratio(nombres[i], nombres[j]))
            if sims:
                min_sim = round(float(min(sims)), 2)
                avg_sim = round(float(sum(sims) / len(sims)), 2)

        n_paises = None
        paises = ""
        if config.col_pais in g.columns:
            vals_pais = g[config.col_pais].dropna().astype(str).map(str.strip).replace("", np.nan).dropna()
            n_paises = int(vals_pais.nunique())
            paises = _colapsar_unicos(vals_pais, config.max_variantes_por_celda)

        flags = []
        if len(g) >= 10:
            flags.append("cluster_grande")
        if min_sim is not None and min_sim < 80:
            flags.append("min_sim_baja")
        if n_paises is not None and n_paises > 1:
            flags.append("multipais_informativo")

        rows.append(
            {
                "ID_GRUPO": gid,
                "CLUSTER_SIZE": int(len(g)),
                "N_VARIANTES": int(len(nombres)),
                "MIN_TOKEN_SET_RATIO": min_sim,
                "AVG_TOKEN_SET_RATIO": avg_sim,
                "N_PAISES_INFORMATIVOS": n_paises,
                "PAISES_INFORMATIVOS": paises,
                "RAZON_SOCIAL_FINAL": _colapsar_unicos(g["RAZON_SOCIAL_FINAL"], 1),
                "VARIANTES_RAZON_SOCIAL": _colapsar_unicos(pd.Series(nombres), config.max_variantes_por_celda),
                "FLAGS_QA": ";".join(flags),
            }
        )

    qa = pd.DataFrame(rows)
    if not qa.empty:
        qa = qa.sort_values(
            ["FLAGS_QA", "CLUSTER_SIZE", "MIN_TOKEN_SET_RATIO"],
            ascending=[False, False, True],
        ).head(config.max_qa_clusters).reset_index(drop=True)
    return qa


def imprimir_resumen(correlativa: pd.DataFrame, golden: pd.DataFrame, conexiones: pd.DataFrame, qa: pd.DataFrame) -> dict[str, Any]:
    n = len(correlativa)
    g = correlativa["ID_GRUPO"].nunique()
    resumen = {
        "registros_entrada_validos": int(n),
        "exportadores_unicos_estimados": int(g),
        "reduccion_pct": round((1 - g / n) * 100, 2) if n else 0.0,
        "clusters_no_triviales": int((correlativa.groupby("ID_GRUPO").size() > 1).sum()),
        "filas_en_clusters_no_triviales": int(len(conexiones)),
        "clusters_qa_generados": int(len(qa)),
    }
    print(json.dumps(resumen, ensure_ascii=False, indent=2))
    print("\nDistribución de tamaños de cluster:")
    print(correlativa.groupby("ID_GRUPO").size().describe().round(2).to_string())
    return resumen


def exportar_resultados(
    config: ConfigFuenteUnica,
    rutas: RutasFuenteUnica,
    correlativa: pd.DataFrame,
    golden: pd.DataFrame,
    conexiones: pd.DataFrame,
    qa: pd.DataFrame,
    resumen: dict[str, Any],
) -> dict[str, Path]:
    """Exporta resultados en CSV/Parquet/Excel según configuración."""
    rutas.exports.mkdir(parents=True, exist_ok=True)
    paths: dict[str, Path] = {}

    if config.exportar_csv:
        paths["correlativa_csv"] = rutas.exports / "fuente_unica_correlativa.csv"
        paths["golden_csv"] = rutas.exports / "fuente_unica_golden_exportadores.csv"
        paths["conexiones_csv"] = rutas.exports / "fuente_unica_conexiones_no_triviales.csv"
        paths["qa_csv"] = rutas.qa / "fuente_unica_qa_clusters.csv"
        correlativa.to_csv(paths["correlativa_csv"], index=False, encoding="utf-8-sig")
        golden.to_csv(paths["golden_csv"], index=False, encoding="utf-8-sig")
        conexiones.to_csv(paths["conexiones_csv"], index=False, encoding="utf-8-sig")
        qa.to_csv(paths["qa_csv"], index=False, encoding="utf-8-sig")

    if config.exportar_parquet:
        paths["correlativa_parquet"] = rutas.exports / "fuente_unica_correlativa.parquet"
        paths["golden_parquet"] = rutas.exports / "fuente_unica_golden_exportadores.parquet"
        paths["conexiones_parquet"] = rutas.exports / "fuente_unica_conexiones_no_triviales.parquet"
        correlativa.to_parquet(paths["correlativa_parquet"], index=False)
        golden.to_parquet(paths["golden_parquet"], index=False)
        conexiones.to_parquet(paths["conexiones_parquet"], index=False)

    if config.exportar_excel:
        paths["excel"] = rutas.exports / "fuente_unica_resultados.xlsx"
        resumen_df = pd.DataFrame([resumen])
        config_df = pd.DataFrame(
            [
                {"parametro": k, "valor": json.dumps(v, ensure_ascii=False, default=str) if isinstance(v, (dict, list, tuple)) else str(v)}
                for k, v in asdict(config).items()
            ]
        )
        with pd.ExcelWriter(paths["excel"], engine="xlsxwriter") as writer:
            resumen_df.to_excel(writer, sheet_name="RESUMEN", index=False)
            config_df.to_excel(writer, sheet_name="CONFIG", index=False)
            golden.to_excel(writer, sheet_name="GOLDEN_EXPORTADORES", index=False)
            correlativa.to_excel(writer, sheet_name="CORRELATIVA", index=False)
            conexiones.to_excel(writer, sheet_name="CONEXIONES", index=False)
            qa.to_excel(writer, sheet_name="QA_CLUSTERS", index=False)

            for sheet_name, df_sheet in {
                "RESUMEN": resumen_df,
                "CONFIG": config_df,
                "GOLDEN_EXPORTADORES": golden,
                "CORRELATIVA": correlativa,
                "CONEXIONES": conexiones,
                "QA_CLUSTERS": qa,
            }.items():
                ws = writer.sheets[sheet_name]
                ws.freeze_panes(1, 0)
                for idx, col in enumerate(df_sheet.columns):
                    width = min(max(len(str(col)) + 2, 12), 45)
                    if len(df_sheet) > 0:
                        sample_width = df_sheet[col].astype(str).str.slice(0, 80).map(len).quantile(0.90)
                        width = min(max(width, int(sample_width) + 2), 55)
                    ws.set_column(idx, idx, width)

    manifest = rutas.exports / "manifest_fuente_unica.json"
    manifest.write_text(
        json.dumps({k: str(v) for k, v in paths.items()}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    paths["manifest"] = manifest
    print("\n📦 Archivos exportados:")
    for k, p in paths.items():
        print(f"  • {k}: {p}")
    return paths



## § B — Ejecución

Esta es la celda que normalmente se edita. Para mantener el pedido original, `usar_pais_en_scoring=False`. Cuando quiera extender el modelo para `PAIS`, cambie ese valor a `True`; el resto del contrato ya está listo.


In [4]:

# ════════════════════════════════════════════════════════════════════════
# CELDA B — CONFIGURAR Y EJECUTAR
# ════════════════════════════════════════════════════════════════════════
_t0_total = time.time()

CONFIG = ConfigFuenteUnica(
    fecha_procesamiento="2026-06-29",
    etiqueta_corrida="FUENTE_UNICA",
    ruta_paquete=Path("/content/drive/MyDrive/ProColombia/record_linkage") if EN_COLAB else Path("/mnt/data/rues_unpacked/rues-linker-main"),
    dir_base=Path("/content/drive/MyDrive/ProColombia/Deduplicación") if EN_COLAB else Path("/mnt/data"),
    dir_datos=Path("/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29") if EN_COLAB else Path("/mnt/data"),
    archivo_excel="Base_exportadores_frutas_a_Col.xlsx",
    hoja_excel=0,
    col_nombre_original="DESC RAZON EXPORTADOR",
    usar_pais_en_scoring=False,  # ← Por pedido: primero SOLO razón social. Cambiar a True para incorporar PAIS.
    smoke_first=True,
    smoke_n=300,
    silenciar_stdout_pipeline=False,
)

RUTAS = RutasFuenteUnica(CONFIG)
RUTAS.crear()
log = setup_logger(RUTAS.logs / "fuente_unica.log")

log.info("Inicio corrida fuente única")
log.info("Memoria inicial: %s", memoria_actual())
print("📁 Carpeta de corrida:", RUTAS.run_dir)
print("📌 Configuración principal:")
print(json.dumps(asdict(CONFIG), ensure_ascii=False, indent=2, default=str))

# 1) Instalar / activar paquete
_t0 = time.time()
REPO = instalar_rues_linker(CONFIG.ruta_paquete)
import record_linkage
print(f"✅ record_linkage {getattr(record_linkage, '__version__', 'sin __version__')} · REPO={REPO}")
fin_paso("B.1 instalar paquete", _t0)

# 2) Configurar perfil runtime auditable
_t0 = time.time()
PERFIL_RUNTIME = configurar_perfil_runtime(CONFIG)
print("🧪 Perfil runtime activo:")
print(json.dumps(PERFIL_RUNTIME, ensure_ascii=False, indent=2, default=str))
fin_paso("B.2 configurar perfil", _t0)

# 3) Cargar y preparar base
_t0 = time.time()
df_fuente = cargar_y_preparar_base(CONFIG, RUTAS)
fin_paso("B.3 cargar base", _t0)

display(df_fuente[[CONFIG.col_nombre_original, CONFIG.col_nombre_estandar, CONFIG.col_nit_tecnico] + ([CONFIG.col_pais] if CONFIG.col_pais in df_fuente.columns else [])].head(10))

# 4) Smoke test controlado
if CONFIG.smoke_first and len(df_fuente) > CONFIG.smoke_n:
    _t0 = time.time()
    log.info("Smoke test con %,d filas", CONFIG.smoke_n)
    smoke_df = df_fuente.sample(CONFIG.smoke_n, random_state=CONFIG.seed).reset_index(drop=True)
    smoke_corr, smoke_conex = ejecutar_deduplicacion(smoke_df, CONFIG, RUTAS.intermedios / "smoke")
    print(
        f"✅ Smoke OK: {len(smoke_corr):,} registros → "
        f"{smoke_corr['ID_GRUPO'].nunique():,} grupos · "
        f"{len(smoke_conex):,} filas en conexiones no triviales"
    )
    del smoke_df, smoke_corr, smoke_conex
    gc.collect()
    fin_paso("B.4 smoke test", _t0)

# 5) Corrida completa
_t0 = time.time()
log.info("Corrida completa con %,d registros válidos", len(df_fuente))
correlativa, conexiones_no_triviales = ejecutar_deduplicacion(
    df_fuente,
    CONFIG,
    RUTAS.intermedios / "full",
)
correlativa["CLUSTER_SIZE"] = correlativa.groupby("ID_GRUPO")["ID_GRUPO"].transform("size")
fin_paso("B.5 deduplicación completa", _t0)

# 6) Golden + QA + resumen
_t0 = time.time()
golden_exportadores = construir_golden_exportadores(correlativa, CONFIG)
qa_clusters = construir_qa_clusters(correlativa, CONFIG)
resumen_corrida = imprimir_resumen(correlativa, golden_exportadores, conexiones_no_triviales, qa_clusters)
fin_paso("B.6 golden y QA", _t0)

# 7) Exportar
_t0 = time.time()
paths_exportados = exportar_resultados(
    CONFIG,
    RUTAS,
    correlativa,
    golden_exportadores,
    conexiones_no_triviales,
    qa_clusters,
    resumen_corrida,
)
fin_paso("B.7 exportar", _t0)

fin_paso("TOTAL", _t0_total)
log.info("Fin corrida. Memoria final: %s", memoria_actual())


2026-06-30 02:27:49 [INFO   ] Inicio corrida fuente única
2026-06-30 02:27:49 [INFO   ] Memoria inicial: proceso=0.16 GB | sistema=8.5% usado | libre=11.60 GB
📁 Carpeta de corrida: /content/drive/MyDrive/ProColombia/Deduplicación/salidas/2026-06-29/FUENTE_UNICA_2026-06-29_20260630_022749
📌 Configuración principal:
{
  "fecha_procesamiento": "2026-06-29",
  "etiqueta_corrida": "FUENTE_UNICA",
  "seed": 42,
  "ruta_paquete": "/content/drive/MyDrive/ProColombia/record_linkage",
  "dir_base": "/content/drive/MyDrive/ProColombia/Deduplicación",
  "dir_datos": "/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29",
  "archivo_excel": "Base_exportadores_frutas_a_Col.xlsx",
  "hoja_excel": 0,
  "nombre_fuente": "EXPORTADORES_FRUTAS_COL",
  "col_nombre_original": "DESC RAZON EXPORTADOR",
  "col_nombre_estandar": "RAZON_SOCIAL",
  "col_nit_tecnico": "NIT",
  "col_pais": "PAIS",
  "cleaning_mode": "BALANCEADO",
  "perfil_base": "deduplication_sin_nit_conservador",
  "perfil_runtime":

,DESC RAZON EXPORTADOR,RAZON_SOCIAL,NIT,PAIS
0,NOGAL FRUITS EXTREMADURA S.L.,NOGAL FRUITS EXTREMADURA S.L.,,España
1,POLAR FRUIT EUROPA S.L,POLAR FRUIT EUROPA S.L,,España
2,POLAR FRUIT EUROPA S.L.,POLAR FRUIT EUROPA S.L.,,España
3,DOLE CHILE S.A.,DOLE CHILE S.A.,,Chile
4,EXPORTADORA FRUTAM LTDA.,EXPORTADORA FRUTAM LTDA.,,Chile
5,FAMILY CROP S.L,FAMILY CROP S.L,,España
6,POLAR FRUIT EUROPA S.L,POLAR FRUIT EUROPA S.L,,Italia
7,VITAOLE SL,VITAOLE SL,,España
8,COPA FRESH S.A.,COPA FRESH S.A.,,España
9,CIREX FRUITS S.C,CIREX FRUITS S.C,,España


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
ValueError: unsupported format character ',' (0x2c) at index 16
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

2026-06-29 21:28:13 | root                      | INFO     | Sistema de Logging configurado exitosamente. Usando hora de Colombia.


2026-06-30 02:28:13 - dedup_preparation - INFO - Procesando nombres con EnhancedTextProcessor...


...

2026-06-29 21:28:24 | GoldenRecordGeneratorV7_Production | INFO     | 💾 Memoria final: 0.3GB


02:28:24 | GoldenRecordGeneratorV7_Production | INFO | 🧹 Recursos liberados


2026-06-29 21:28:24 | GoldenRecordGeneratorV7_Production | INFO     | 🧹 Recursos liberados
2026-06-29 21:28:24 | record_linkage.pipeline.linkage_pipeline | INFO     | ✅ Completado: Fase 4: Golden Records en 0.86s
2026-06-29 21:28:24 | record_linkage.pipeline.linkage_pipeline | INFO     |    📈 Estado Memoria Final: 0.3GB usados (10.9% del sistema) | Consumo en fase: +2.1 MB | Salud del Sistema: 🟢 NORMAL
🔗 Iniciando consolidación BALANCEADA por NIT (v4 CORREGIDA)...
   • Golden records: 750
   • Correlativa: 1,607
   • Usando TextProcessor con fallback de 3 niveles
   • NITs únicos: 0
   • NITs en múltiples grupos: 0
   ✓ No hay NITs duplicados
2026-06-29 21:28:24 | record_linkage.pipeline.linkage_pipeline | INFO     | ▶️  Iniciando: Fase 5: Reportes y Exportación
2026-06-29 21:28:24 | record_linkage.pipeline.linkage_pipeline | INFO     |     💾 Estado Memoria Inicial: 0.3GB usados por el proceso (10.9% del sistema)
2026-06-29 21:28:24 | record_linkage.exporters.smart | INFO     | ▶️  Ini

2026-06-30 02:28:27 - deduplicate_unified - INFO - Deduplicación completada en 7.5s. Archivos guardados: correlativa_unificada.parquet, conexiones_no_triviales.parquet


2026-06-29 21:28:27 | deduplicate_unified       | INFO     | Deduplicación completada en 7.5s. Archivos guardados: correlativa_unificada.parquet, conexiones_no_triviales.parquet

⏱️  B.5 deduplicación completa: 7.52 s
{
  "registros_entrada_validos": 1607,
  "exportadores_unicos_estimados": 750,
  "reduccion_pct": 53.33,
  "clusters_no_triviales": 286,
  "filas_en_clusters_no_triviales": 1143,
  "clusters_qa_generados": 200
}

Distribución de tamaños de cluster:
count    750.00
mean       2.14
std        2.42
min        1.00
25%        1.00
50%        1.00
75%        2.00
max       22.00

⏱️  B.6 golden y QA: 5.69 s

📦 Archivos exportados:
  • correlativa_csv: /content/drive/MyDrive/ProColombia/Deduplicación/salidas/2026-06-29/FUENTE_UNICA_2026-06-29_20260630_022749/exports/fuente_unica_correlativa.csv
  • golden_csv: /content/drive/MyDrive/ProColombia/Deduplicación/salidas/2026-06-29/FUENTE_UNICA_2026-06-29_20260630_022749/exports/fuente_unica_golden_exportadores.csv
  • conexiones_cs


## § C — Inspección rápida de resultados

Use esta sección para detectar sobre-fusión. Sin ground truth etiquetado, no se puede afirmar F1; lo responsable es revisar clusters grandes, clusters multipaís y clusters con baja similitud mínima.


In [5]:

# ════════════════════════════════════════════════════════════════════════
# CELDA C — QA visual rápido
# ════════════════════════════════════════════════════════════════════════

print("=== RESUMEN ===")
print(json.dumps(resumen_corrida, ensure_ascii=False, indent=2))

print("\n=== TOP 20 CLUSTERS POR TAMAÑO ===")
top_clusters = (
    correlativa.groupby("ID_GRUPO")
    .agg(
        CLUSTER_SIZE=("ID_GRUPO", "size"),
        RAZON_SOCIAL_FINAL=("RAZON_SOCIAL_FINAL", "first"),
        VARIANTES=(CONFIG.col_nombre_original, lambda x: _colapsar_unicos(x, 8)),
        PAISES=(CONFIG.col_pais, lambda x: _colapsar_unicos(x, 8)) if CONFIG.col_pais in correlativa.columns else ("ID_GRUPO", "size"),
    )
    .sort_values("CLUSTER_SIZE", ascending=False)
    .head(20)
    .reset_index()
)
display(top_clusters)

print("\n=== QA CLUSTERS CRÍTICOS / REVISABLES ===")
if qa_clusters.empty:
    print("No hay clusters no triviales para QA.")
else:
    display(qa_clusters.head(50))

print("\n=== MUESTRA DE CORRELATIVA ===")
cols = [
    "ID_GRUPO",
    "CLUSTER_SIZE",
    CONFIG.col_nombre_original,
    "RAZON_SOCIAL_FINAL",
    "NOMBRE_LIMPIO",
]
if CONFIG.col_pais in correlativa.columns:
    cols.append(CONFIG.col_pais)
display(correlativa.sort_values(["CLUSTER_SIZE", "ID_GRUPO"], ascending=[False, True])[cols].head(100))


=== RESUMEN ===
{
  "registros_entrada_validos": 1607,
  "exportadores_unicos_estimados": 750,
  "reduccion_pct": 53.33,
  "clusters_no_triviales": 286,
  "filas_en_clusters_no_triviales": 1143,
  "clusters_qa_generados": 200
}

=== TOP 20 CLUSTERS POR TAMAÑO ===


,ID_GRUPO,CLUSTER_SIZE,RAZON_SOCIAL_FINAL,VARIANTES,PAISES
0,1,22,POLAR FRUIT EUROPA S.L,POLAR FRUIT EUROPA S.L | POLAR FRUIT EUROPA S....,España | Francia | Italia | Polonia | Portugal
1,3,21,DOLE CHILE S.A.,DOLE CHILE S.A. | DOLE CHILE S. A. | DOLE CH...,Chile | Perú
2,87,16,"SUN FRESH INTERNATIONAL, LLC",SUN FRESH EXPORT LLC | SUN FRESH INTERNACIONAL...,Estados Unidos | Perú
3,303,15,COMERCIALIZADORA DE ALIMENTOS LOADS SPA,COMERCIALIZADORA DE ALIMENTOS LOADS SPA | COM...,Alemania | Chile | Estados Unidos | Italia | P...
4,56,15,PARAMOUNT EXPORT COMPANY,PARAMOUNT EXPORT COMPANY,Argentina | Canadá | Chile | Estados Unidos | ...
5,363,15,TERRA EXPORTS LLC,TERRA EXPORTS | TERRA EXPORTS LLC | TERRA EXP...,Chile | Estados Unidos | Francia | Italia | Perú
6,48,14,ROBINSON FRESH CHILE S.A,ROBINSON FRESH CHILE S A | ROBINSON FRESH CHIL...,Chile | Perú
7,35,13,EXPORTADORA Y COMERCIALIZADORA ERFRUT LTDA,EXPORTADORA Y COMERCIALIZADORA ERFRUT LTDA | E...,Argentina | Chile | Estados Unidos | Italia | ...
8,77,13,FRUTERA PERU SAC,FRUTERA PERU S.A.C | FRUTERA PERU S.A.C. | FRU...,Chile | Francia | Perú
9,22,13,MITEXPORT SPA,MITEXPORT SPA,Argentina | Chile | España | Francia | Italia ...



=== QA CLUSTERS CRÍTICOS / REVISABLES ===


,ID_GRUPO,CLUSTER_SIZE,N_VARIANTES,MIN_TOKEN_SET_RATIO,AVG_TOKEN_SET_RATIO,N_PAISES_INFORMATIVOS,PAISES_INFORMATIVOS,RAZON_SOCIAL_FINAL,VARIANTES_RAZON_SOCIAL,FLAGS_QA
0,96,9,3,93.33,95.55,4,Chile | España | Polonia | Portugal,"TRIDGE CO., LTD","TRIDGE CO., LTD | TRIDGE CO., LTD. | TRIDGE CO...",multipais_informativo
1,20,9,2,98.59,98.59,3,Chile | Estados Unidos | Perú,ASESORIAS E INVERSIONES ITAHUE LTDA,ASESORIAS E INVERSIONES ITAHUE LTDA | ASESORIA...,multipais_informativo
2,194,8,2,96.97,96.97,6,Chile | España | Francia | Italia | Polonia | ...,BLUE WHALE S.A.S,BLUE WHALE S.A.S | BLUE WHALE S.A.S.,multipais_informativo
3,88,8,2,100.00,100.00,4,España | Francia | Italia | Polonia,DISTRIMEX SAS,DISTRIMEX SAS | DISTRIMEX SAS,multipais_informativo
4,127,8,3,100.00,100.00,2,Chile | Italia,FRUTERA AGROSAN EXPORT SPA,FRUTERA AGROSAN EXPORT SPA | FRUTERA AGROSAN ...,multipais_informativo
5,42,8,1,NaN,NaN,4,Chile | España | Estados Unidos | Portugal,EXPORTADORA METRO SA.,EXPORTADORA METRO SA.,multipais_informativo
6,334,7,2,98.82,98.82,3,Chile | Estados Unidos | Perú,SERVICIOS AGRICOLAS OPPENHEIMER CHILE LTDA,SERVICIOS AGRICOLAS OPPENHEIMER CHILE LTDA | S...,multipais_informativo
7,17,7,2,100.00,100.00,2,Chile | Perú,COMERCIAL E INVERSIONES CMX CHILE SPA,CMX CHILE SPA | COMERCIAL E INVERSIONES CMX CH...,multipais_informativo
8,283,7,1,NaN,NaN,5,Argentina | Chile | Estados Unidos | Francia |...,SALIX FRUITS LLC,SALIX FRUITS LLC,multipais_informativo
9,263,6,4,90.91,93.94,2,Italia | Polonia,B&B FRUTTA S.R.L.,B & B FRUTTA S.R.L | B & B FRUTTA S.R.L. | B&B...,multipais_informativo



=== MUESTRA DE CORRELATIVA ===


,ID_GRUPO,CLUSTER_SIZE,DESC RAZON EXPORTADOR,RAZON_SOCIAL_FINAL,NOMBRE_LIMPIO,PAIS
1,1,22,POLAR FRUIT EUROPA S.L,POLAR FRUIT EUROPA S.L,EUROPA,España
2,1,22,POLAR FRUIT EUROPA S.L.,POLAR FRUIT EUROPA S.L,EUROPA,España
6,1,22,POLAR FRUIT EUROPA S.L,POLAR FRUIT EUROPA S.L,EUROPA,Italia
38,1,22,POLAR FRUIT EUROPA S.L.,POLAR FRUIT EUROPA S.L,EUROPA,Italia
105,1,22,POLAR FRUIT EUROPA S.L.,POLAR FRUIT EUROPA S.L,EUROPA,España
...,...,...,...,...,...,...
638,363,15,TERRA EXPORTS LLC,TERRA EXPORTS LLC,TERRA EXPORTS,Chile
707,363,15,TERRA EXPORTS,TERRA EXPORTS LLC,TERRA EXPORTS,Italia
974,363,15,TERRA EXPORTS LLC,TERRA EXPORTS LLC,TERRA EXPORTS,Chile
1179,363,15,TERRA EXPORTS LLC,TERRA EXPORTS LLC,TERRA EXPORTS,Chile



## § D — Extensión controlada para `PAIS`

No ejecute esto si quiere respetar estrictamente la primera versión por nombre. Esta celda documenta cómo activar `PAIS` sin reescribir el notebook.


In [6]:

# ════════════════════════════════════════════════════════════════════════
# CELDA D — Cómo extender a PAIS (documentación ejecutable opcional)
# ════════════════════════════════════════════════════════════════════════

print("Para incorporar PAIS en el scoring, cambie en la celda B:")
print("    usar_pais_en_scoring=True")
print("\nEl contrato ya está definido como:")
print(json.dumps(CONFIG.extra_features_pais, ensure_ascii=False, indent=2, default=str))
print(
    "\nInterpretación: categorical_signed premia PAIS igual y penaliza PAIS distinto. "
    "Úselo solo si conceptualmente el mismo exportador no debería agruparse entre países."
)


Para incorporar PAIS en el scoring, cambie en la celda B:
    usar_pais_en_scoring=True

El contrato ya está definido como:
[
  {
    "column": "PAIS",
    "weight": 0.2,
    "type": "categorical_signed"
  }
]

Interpretación: categorical_signed premia PAIS igual y penaliza PAIS distinto. Úselo solo si conceptualmente el mismo exportador no debería agruparse entre países.



## § E — Tiempos de ejecución


In [7]:

# ════════════════════════════════════════════════════════════════════════
# CELDA E — Tiempos
# ════════════════════════════════════════════════════════════════════════

print("=" * 72)
print("⏱️  TIEMPOS DE LA CORRIDA")
print("=" * 72)
for nombre, segundos in TIEMPOS.items():
    print(f"{nombre:<34} {segundos:>10,.2f} s")
print("=" * 72)


⏱️  TIEMPOS DE LA CORRIDA
B.1 instalar paquete                    23.84 s
B.2 configurar perfil                    0.00 s
B.3 cargar base                          0.66 s
B.4 smoke test                           5.89 s
B.5 deduplicación completa               7.52 s
B.6 golden y QA                          5.69 s
B.7 exportar                             1.64 s
TOTAL                                   45.33 s
